# 01 — Explore the data

**Goal:** look at real and fake face photos, count how many of each we have, and check the image sizes.
Nothing is trained or saved here.

**Dataset:** Kaggle `xhlulu/140k-real-and-fake-faces` — real faces come from FFHQ, fake faces from StyleGAN.
More details in `docs/03_data.md` in the project.

**Labels:** `fake` = 1 (what we want to catch), `real` = 0.

**How to use:** this notebook is made for **Kaggle**. Do the settings below once, then run the cells from top to
bottom (Shift+Enter). No GPU is needed.

### Before you start: Kaggle settings

Open the right-hand panel of the notebook editor (the **>|** icon at the top right if it is hidden):

- **Settings → Internet → On.** Step 1 downloads the project code from GitHub. Kaggle asks you to verify your phone once.
- **Add Input → Datasets →** search **"140k Real and Fake Faces"** (by xhlulu) → **+**. These are the images.
  Kaggle attaches them read-only under `/kaggle/input/`, so nothing is downloaded or unzipped.

After you change a setting, Kaggle restarts the session: run the cells again from the top.

## Step 1 — Get the project and find the images

**What this does, in plain words:**

1. Loads the Python tools we need.
2. Downloads the project code (`src/`, `configs/`) from GitHub into `/kaggle/working/`, the only folder on Kaggle
   where we can write files. If Internet is off, it tells you so in plain words.
3. Finds the dataset you attached with **Add Input** somewhere under `/kaggle/input/`.
4. Counts the images in each folder and prints `Dataset ready.`

The attached dataset has this layout:

```
/kaggle/input/.../real_vs_fake/real-vs-fake/
├── train/real/*.jpg   train/fake/*.jpg     100,000 photos: the model learns from these
├── valid/real/*.jpg   valid/fake/*.jpg      20,000 photos: used to choose the best model
└── test/real/*.jpg    test/fake/*.jpg       20,000 photos: kept for the final score
```

In [ ]:
# 1. Tools we need
import os
import random
import subprocess
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image

# 2. Get the project code (src/ and configs/) from GitHub.
#    It goes into /kaggle/working: the only folder on Kaggle where we can write files.
REPO_URL = "https://github.com/msondhi/UOT_SCS_3546_041_Deep-Learning.git"
PROJECT_ROOT = Path("/kaggle/working/UOT_SCS_3546_041_Deep-Learning")

if not PROJECT_ROOT.exists():
    result = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_ROOT)])
    if result.returncode != 0:
        raise RuntimeError("Could not download the project from GitHub. "
                           "Turn on Settings -> Internet in the right-hand panel, then run this cell again.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))   # lets Python import the project's own code
print("Project folder:", PROJECT_ROOT)

# 3. Find the dataset you attached with Add Input. Kaggle puts it somewhere under /kaggle/input.
def find_dataset():
    """Look inside /kaggle/input for the folder called real-vs-fake that has train/ valid/ test/."""
    for depth in range(1, 5):                                  # look 1, 2, 3, then 4 folders deep
        pattern = "/".join(["*"] * depth) + "/real-vs-fake"    # e.g. "*/*/real-vs-fake"
        for folder in Path("/kaggle/input").glob(pattern):
            if (folder / "train").is_dir():
                return folder
    return None

DATA_ROOT = find_dataset()       # folder with train/ valid/ test/, or None if not attached
if DATA_ROOT is None:
    print('Dataset not found: Add Input -> Datasets -> "140k Real and Fake Faces", then run this cell again.')
else:
    print("Images folder:", DATA_ROOT)

SPLITS = ["train", "valid", "test"]
IMAGE_LISTS = {}   # remembers the file list of each folder, so every folder is read only once

def list_images(split, class_name):
    """All .jpg files in one folder, e.g. list_images("train", "fake").

    The first call reads the folder (a few seconds for 50,000 files); after that the list
    is remembered in IMAGE_LISTS and every later call is instant.
    """
    if DATA_ROOT is None:
        return []
    folder = DATA_ROOT / split / class_name
    if folder not in IMAGE_LISTS:
        names = sorted(os.listdir(folder)) if folder.is_dir() else []    # sorting plain names is fast
        IMAGE_LISTS[folder] = [folder / name for name in names if name.lower().endswith(".jpg")]
    return IMAGE_LISTS[folder]

# 4. Read the shared project settings (image size, ...) and count the images.
from deepfake_kyc.config import load_config

cfg = load_config(PROJECT_ROOT / "configs" / "base.yaml")

DATA_AVAILABLE = True
for split in SPLITS:
    n_real, n_fake = len(list_images(split, "real")), len(list_images(split, "fake"))
    print(f"{split:6} real={n_real:,}  fake={n_fake:,}")
    if n_real == 0 or n_fake == 0:
        DATA_AVAILABLE = False      # every split needs both real and fake images
print("Dataset ready." if DATA_AVAILABLE else "Dataset missing: see the message above.")

## Step 2 — Count real and fake images

**What this does:** counts the real and fake images in each split and draws a bar chart.

**Why:** a big difference between the two bars (class imbalance) or an empty folder would be a problem for training.

- `train` teaches the model, `valid` helps choose the best model, `test` is kept for the final score only.

In [ ]:
if DATA_AVAILABLE:
    real_counts = [len(list_images(split, "real")) for split in SPLITS]
    fake_counts = [len(list_images(split, "fake")) for split in SPLITS]

    for split, n_real, n_fake in zip(SPLITS, real_counts, fake_counts):
        print(f"{split:6} total={n_real + n_fake:,} | real (0)={n_real:,} | fake (1)={n_fake:,}")

    positions = range(len(SPLITS))
    plt.figure(figsize=(6, 3.5))
    plt.bar([x - 0.2 for x in positions], real_counts, width=0.4, label="real (0)")
    plt.bar([x + 0.2 for x in positions], fake_counts, width=0.4, label="fake (1)")
    plt.xticks(positions, SPLITS)
    plt.ylabel("images")
    plt.title("Class balance per split")
    plt.legend()
    plt.show()
else:
    print("Skipped: dataset not found (Step 1).")

## Step 3 — Check original image sizes

**What this does:** opens 200 random training images and counts their `(width, height)`.

**Why:** the training code resizes every image to one square size (`224×224`) before the model sees it.
It is good to know the original size first. 200 images are enough to see the pattern without opening all 100,000.

In [ ]:
if DATA_AVAILABLE:
    random.seed(0)   # same random picks every time you run the cell
    train_images = list_images("train", "real") + list_images("train", "fake")
    sample = random.sample(train_images, min(200, len(train_images)))

    sizes = Counter()
    for path in sample:
        with Image.open(path) as image:
            sizes[image.size] += 1

    print(f"Checked {len(sample)} training images.")
    print("(width, height) -> how many:")
    for size, count in sizes.most_common():
        print(f"  {size}: {count}")
    image_size = cfg["data"]["image_size"]
    print()
    print(f"The training code resizes every image to {image_size}x{image_size}.")
else:
    print("Skipped: dataset not found (Step 1).")

## Step 4 — Look at a few examples

**What this does:** shows 4 random real and 4 random fake training faces side by side.

**Why:** a quick check for broken files or wrong labels. Fakes are often hard to spot by eye —
that is why we need a model. A few pictures say nothing about how well a model will work.

In [ ]:
if DATA_AVAILABLE:
    random.seed(0)
    reals = random.sample(list_images("train", "real"), 4)
    fakes = random.sample(list_images("train", "fake"), 4)
    examples = [(path, "real") for path in reals] + [(path, "fake") for path in fakes]

    fig, axes = plt.subplots(1, len(examples), figsize=(2.2 * len(examples), 2.6))
    for ax, (path, label) in zip(axes, examples):
        with Image.open(path) as image:
            ax.imshow(image.convert("RGB"))
        ax.set_title(label)
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Skipped: dataset not found (Step 1).")

## Next

Notebook **02_degradations_gallery** shows what these faces look like after the robustness degradations
(blur, JPEG, downscale, noise, screen capture). It attaches the same dataset itself; it does not need this notebook.

## Quick notes to remember

- `train` teaches the model; `valid` helps choose the model; `test` is held back for the final score.
- `fake = 1` is the positive class and `real = 0` is the negative class.
- The size check looks at only 200 training images, not the whole dataset.
- The example faces are only a sanity check. Do not judge detection accuracy from a few pictures.
- Write the real class counts and image sizes from your run into your report.